In [1]:
# import general modules
import os, sys
import torch, math
import torch.nn as nn
import numpy as np


# add custom modules folder to Python's search path
sys.path.append(os.path.abspath('../../modules'))
sys.path.append(os.path.abspath('../../modules/mnist'))


# load custom modules
import vae_fu as vfu
import vae_train as vt
import vae_ortho as vo   # module for UNO (UNlearning with Orthogonalization)
import vae_surgery as vs # module for S (gradient Surgery)
import vae_os as vos     # module for UNO-S (alternative UNO and Surgery)
import vae_ascent as va  # module for A (gradient Ascent)
import vae_ad as vad     # module for A-D (Ascent-Descent)
import classifier as cl  # architecture and loader for pre-trained classifiers
import batch as bt       # module for running experiments in bulk
import utility as ut     # some helper functions
from vae import VAE      # architecture for VAE
import datapipe          # custom dataloaders
 

# set general parameters
device = ut.get_device() # find available device type, CUDA is preferable
mnist_folder = "../../data/MNIST" # folder containing pre-trained VAE and classifiers in subfolders 
experiment_folder = "../../data/MNIST/MNIST-Experiments" # folder where the results of the experiments will be stored
num_fid_samples = 25000 # sample size for calculating FID scores
num_experiments = 10 # number of runs for unlearning algorithm
dl_r, dl_f = datapipe.MNIST().get_dataloader_rf(128)
model0_path = f"{mnist_folder}/vae/vae_200.pth"
model = VAE(device=device)
model.load_state_dict(torch.load(model0_path))
model.to(device)



res = vfu.compute_direction_and_threshold_from_dataloaders(model, dl_f, dl_r, device, use_mu=True)
z_e = res['feature_direction'].to(device)
v_unit = res['feature_direction_unit'].to(device)
delta = res['delta']

In [4]:
z = torch.randn(size=(100, 2)).to(device)
vfu.L_full(model, model, z, z_e, v_unit, delta)

tensor(239.2395, device='mps:0', grad_fn=<AddBackward0>)

In [4]:
model.decoder(z).shape

torch.Size([100, 784])